# WBCBench2026 Submission Notebook

This notebook trains a PyTorch CNN model on the WBCBench2026 dataset, evaluates on validation, and generates predictions for submission.

## 1. Import Required Libraries

In [1]:
import torch
from torch import nn
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from pathlib import Path
from tqdm.auto import tqdm
import csv
from PIL import Image
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import pandas as pd

c:\Documents\Kings college London\AI\Research\white blood cell classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load the Competition Dataset

In [2]:
# Set data path - change for Kaggle
BASE_DIR = Path(r"C:\Documents\Kings college London\AI\Research\white blood cell classification\wbc-bench-2026")
# For Kaggle: 
# BASE_DIR = Path('/kaggle/input/wbc-bench-2026/wbc-bench-2026')

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load CSVs
train_df = pd.read_csv(BASE_DIR / "phase2_train.csv")
val_df = pd.read_csv(BASE_DIR / "phase2_eval.csv")
test_df = pd.read_csv(BASE_DIR / "phase2_test.csv")

# Add folder column
train_df['folder'] = 'train'
val_df['folder'] = 'eval'
test_df['folder'] = 'test'

from sklearn.model_selection import train_test_split

# Combine train and eval for training
combined_train_df = pd.concat([train_df, val_df], ignore_index=True)

# Split combined into train and val (80-20)
train_df, val_df = train_test_split(combined_train_df, test_size=0.2, stratify=combined_train_df['labels'], random_state=42)

## 3. Explore the Data

In [3]:
print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)
print("Test shape:", test_df.shape)
print("Unique labels in train:", train_df['labels'].unique())
print("Unique labels in val:", val_df['labels'].unique())
print("Test labels (should be empty):", test_df['labels'].unique())

# Define class names from train labels
class_names = sorted(train_df['labels'].unique())
num_classes = len(class_names)
print("Number of classes:", num_classes)
print("Class names:", class_names)

Train shape: (24197, 4)
Val shape: (6050, 4)
Test shape: (16477, 3)
Unique labels in train: <StringArray>
['SNE',  'LY',  'MO',  'BL',  'EO', 'MMY', 'VLY', 'BNE',  'MY', 'PMY',  'BA',
  'PC', 'PLY']
Length: 13, dtype: str
Unique labels in val: <StringArray>
['SNE',  'LY', 'MMY',  'BL',  'BA',  'MO',  'MY', 'BNE',  'EO', 'PLY', 'VLY',
  'PC', 'PMY']
Length: 13, dtype: str
Test labels (should be empty): [nan]
Number of classes: 13
Class names: ['BA', 'BL', 'BNE', 'EO', 'LY', 'MMY', 'MO', 'MY', 'PC', 'PLY', 'PMY', 'SNE', 'VLY']


## 4. Preprocess the Data

In [4]:
data_transform = transforms.Compose([
    transforms.Resize(size=(224, 224)),  # ResNet expects 224x224
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(size=(224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [5]:
class WBCImageDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None, is_test=False):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = self.dataframe.iloc[idx]['ID']
        folder = self.dataframe.iloc[idx]['folder']
        img_path = BASE_DIR / "phase2" / folder / img_name  # ID includes .jpg extension
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        if self.is_test:
            return image, img_name
        else:
            label_str = self.dataframe.iloc[idx]['labels']
            label = class_names.index(label_str)
            return image, label

In [6]:
# Create datasets
train_dataset = WBCImageDataset(train_df, BASE_DIR / "phase2" / "train", transform=data_transform)
val_dataset = WBCImageDataset(val_df, BASE_DIR / "phase2" / "eval", transform=val_transform)
test_dataset = WBCImageDataset(test_df, BASE_DIR / "phase2" / "test", transform=val_transform, is_test=True)

# Create data loaders
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 5. Build the Model

In [7]:
from torchvision import models
from torchvision.models import ResNet50_Weights

class WBCClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.model = models.resnet50(weights=ResNet50_Weights.DEFAULT)
        # Freeze early layers
        for param in self.model.parameters():
            param.requires_grad = False
        # Unfreeze the last few layers
        for param in self.model.layer2.parameters():
            param.requires_grad = True
        for param in self.model.layer3.parameters():
            param.requires_grad = True
        for param in self.model.layer4.parameters():
            param.requires_grad = True
        # Replace classifier
        self.model.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(self.model.fc.in_features, num_classes)
        )

    def forward(self, x):
        return self.model(x)

model = WBCClassifier(num_classes=len(class_names)).to(device)

In [8]:
# Calculate class weights for imbalance
class_counts = train_df['labels'].value_counts().reindex(class_names)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * len(class_names)
class_weights = torch.tensor(class_weights.values, dtype=torch.float).to(device)

loss_fn = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)

# Scheduler
EPOCHS = 10
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# Results tracking
results = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "val_macro_f1": []}

## 6. Train the Model

In [9]:
# Training loop
for epoch in tqdm(range(EPOCHS)):
    model.train()
    train_loss, train_acc = 0, 0
    
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        y_pred = model(X)
        loss = loss_fn(y_pred, y)
        train_loss += loss.item()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        y_pred_class = torch.argmax(y_pred, dim=1)
        train_acc += (y_pred_class == y).sum().item() / len(y_pred)
    
    train_loss /= len(train_loader)
    train_acc /= len(train_loader)
    
    # Validation
    model.eval()
    val_loss, val_acc = 0, 0
    val_preds, val_labels = [], []
    
    with torch.inference_mode():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            y_pred = model(X)
            loss = loss_fn(y_pred, y)
            val_loss += loss.item()
            
            y_pred_class = torch.argmax(y_pred, dim=1)
            val_acc += (y_pred_class == y).sum().item() / len(y_pred)
            
            val_preds.extend(y_pred_class.cpu().numpy())
            val_labels.extend(y.cpu().numpy())
    
    val_loss /= len(val_loader)
    val_acc /= len(val_loader)
    val_macro_f1 = f1_score(val_labels, val_preds, average='macro')
    
    results["train_loss"].append(train_loss)
    results["train_acc"].append(train_acc)
    results["val_loss"].append(val_loss)
    results["val_acc"].append(val_acc)
    results["val_macro_f1"].append(val_macro_f1)
    
    print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2%}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2%}, Val Macro-F1: {val_macro_f1:.4f}")
    
    scheduler.step()

 10%|█         | 1/10 [07:13<1:05:01, 433.48s/it]

Epoch 1: Train Loss: 1.4750, Train Acc: 69.72%, Val Loss: 0.9535, Val Acc: 82.32%, Val Macro-F1: 0.5291


 20%|██        | 2/10 [11:18<43:00, 322.57s/it]  

Epoch 2: Train Loss: 0.8767, Train Acc: 80.29%, Val Loss: 0.8132, Val Acc: 80.67%, Val Macro-F1: 0.5391


 30%|███       | 3/10 [15:05<32:32, 278.92s/it]

...

Epoch 8: Train Loss: 0.3091, Train Acc: 90.61%, Val Loss: 0.7384, Val Acc: 88.06%, Val Macro-F1: 0.6726


 90%|█████████ | 9/10 [38:55<04:05, 245.65s/it]

Epoch 9: Train Loss: 0.2699, Train Acc: 91.70%, Val Loss: 0.7527, Val Acc: 87.29%, Val Macro-F1: 0.6849


100%|██████████| 10/10 [43:51<00:00, 263.16s/it]

Epoch 10: Train Loss: 0.2395, Train Acc: 91.86%, Val Loss: 0.7207, Val Acc: 88.75%, Val Macro-F1: 0.7003


## 7. Make Predictions on Test Data

In [10]:
model.eval()
predictions = []
ids = []
with torch.inference_mode():
    for X, img_ids in test_loader:
        X = X.to(device)
        preds = model(X)
        pred_labels = preds.argmax(dim=1).cpu().numpy()
        predictions.extend(pred_labels)
        ids.extend(img_ids)

# Map back to class names
pred_class_names = [class_names[pred] for pred in predictions]

## 8. Create Submission File

In [11]:
submission_df = pd.DataFrame({
    'ID': ids,
    'labels': pred_class_names
})

## 9. Save Submission File

In [13]:
# For local testing
submission_df.to_csv('submission.csv', index=False)

# For Kaggle
# submission_df.to_csv('/kaggle/working/submission.csv', index=False)

print("Submission file saved as submission.csv")

Submission file saved as submission.csv
